In [1]:
import json
import re
import shutil
import subprocess
import tempfile
from pathlib import Path

import httpx
import pandas as pd

In [2]:
client = httpx.Client(
    timeout=httpx.Timeout(connect=5.0, read=30.0, write=30.0, pool=30.0),
    limits=httpx.Limits(max_keepalive_connections=0, max_connections=1),
    trust_env=False,
    follow_redirects=True,
    headers={
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36"
        ),
    },
)

## 巨潮资讯网公告检索

巨潮这里验证的是**公司公告原始数据**，不是新浪那种已经整理好的分红、配股或复权因子表。请求顺序很直接：先用股票代码请求 `topSearch/query` 得到 `orgId`，再用股票代码和 `orgId` 请求 `hisAnnouncement/query`，最后用返回记录里的 `adjunctUrl` 下载 PDF 或 HTML 公告。

普通的 `10 派 1`、`10 送 1`、配股比例、配股价格和日期，可以从权益分派实施公告或配股公告中读取。差异化分红、股权分置、回购注销、发行股份等特殊行为，则需要继续打开公告原文读取具体数字。这个 notebook 只按步骤请求和展示数据，不定义通用函数，也不做多数据源校验。

`seDate` 使用公告日期范围，不是除权除息日范围；本例固定到 `2026-08-29`，方便和当前调研记录对应。

In [3]:
CNINFO_TOP_SEARCH_ENDPOINT = (
    "https://www.cninfo.com.cn/new/information/topSearch/query"
)
CNINFO_ANNOUNCEMENT_ENDPOINT = (
    "https://www.cninfo.com.cn/new/hisAnnouncement/query"
)
CNINFO_STATIC_BASE_URL = "https://static.cninfo.com.cn/"
CNINFO_AS_OF_DATE = "2026-08-29"
CNINFO_DATE_RANGE = f"1990-01-01~{CNINFO_AS_OF_DATE}"
CNINFO_REQUEST_HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://www.cninfo.com.cn",
    "Referer": "https://www.cninfo.com.cn/",
    "User-Agent": client.headers["User-Agent"],
}
CNINFO_PDF_HEADERS = {
    "Accept": "application/pdf,text/html,application/xhtml+xml,*/*;q=0.8",
    "Referer": "https://www.cninfo.com.cn/",
    "User-Agent": client.headers["User-Agent"],
}
CNINFO_PDF_TEXT_AVAILABLE = shutil.which("pdftotext") is not None

### 通过股票代码获取 `orgId`

`topSearch/query` 返回的是列表。这里把新浪 notebook 中不同股票的逐步请求方式原样展开，避免隐藏股票代码和组织 ID 的对应关系。

In [4]:
cninfo_603888_top_resp = client.post(
    CNINFO_TOP_SEARCH_ENDPOINT,
    data={"keyWord": "603888", "maxNum": 10},
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_603888_top_resp.raise_for_status()
cninfo_603888_top_json = cninfo_603888_top_resp.json()
cninfo_603888_top_json

[{'code': '603888',
  'pinyin': 'xhw',
  'sjstsBond': 'false',
  'category': 'A股',
  'type': 'shj',
  'delisted': 'false',
  'orgId': '9900024951',
  'zwjc': '新华网'}]

In [5]:
cninfo_600081_top_resp = client.post(
    CNINFO_TOP_SEARCH_ENDPOINT,
    data={"keyWord": "600081", "maxNum": 10},
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600081_top_resp.raise_for_status()
cninfo_600081_top_json = cninfo_600081_top_resp.json()
cninfo_600081_top_json

[{'code': '600081',
  'pinyin': 'dfkj',
  'sjstsBond': 'false',
  'category': 'A股',
  'type': 'shj',
  'delisted': 'false',
  'orgId': 'gssh0600081',
  'zwjc': '东风科技'}]

In [6]:
cninfo_688603_top_resp = client.post(
    CNINFO_TOP_SEARCH_ENDPOINT,
    data={"keyWord": "688603", "maxNum": 10},
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_688603_top_resp.raise_for_status()
cninfo_688603_top_json = cninfo_688603_top_resp.json()
cninfo_688603_top_json

[{'code': '688603',
  'pinyin': 'tckj',
  'sjstsBond': 'false',
  'category': 'A股',
  'type': 'shj',
  'delisted': 'false',
  'orgId': '9900048047',
  'zwjc': '天承科技'}]

In [7]:
cninfo_600519_top_resp = client.post(
    CNINFO_TOP_SEARCH_ENDPOINT,
    data={"keyWord": "600519", "maxNum": 10},
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600519_top_resp.raise_for_status()
cninfo_600519_top_json = cninfo_600519_top_resp.json()
cninfo_600519_top_json

[{'code': '600519',
  'pinyin': 'gzmt',
  'sjstsBond': 'false',
  'category': 'A股',
  'type': 'shj',
  'delisted': 'false',
  'orgId': 'gssh0600519',
  'zwjc': '贵州茅台'}]

In [8]:
cninfo_600497_top_resp = client.post(
    CNINFO_TOP_SEARCH_ENDPOINT,
    data={"keyWord": "600497", "maxNum": 10},
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600497_top_resp.raise_for_status()
cninfo_600497_top_json = cninfo_600497_top_resp.json()
cninfo_600497_top_json

[{'code': '600497',
  'pinyin': 'chxz',
  'sjstsBond': 'false',
  'category': 'A股',
  'type': 'shj',
  'delisted': 'false',
  'orgId': 'gssh0600497',
  'zwjc': '驰宏锌锗'}]

In [9]:
cninfo_000539_top_resp = client.post(
    CNINFO_TOP_SEARCH_ENDPOINT,
    data={"keyWord": "000539", "maxNum": 10},
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_000539_top_resp.raise_for_status()
cninfo_000539_top_json = cninfo_000539_top_resp.json()
cninfo_000539_top_json

[{'code': '000539',
  'pinyin': 'ydla',
  'sjstsBond': 'false',
  'category': 'A股',
  'type': 'shj',
  'delisted': 'false',
  'orgId': 'gssz0000539',
  'zwjc': '粤电力Ａ'}]

In [10]:
cninfo_603888_org_id = cninfo_603888_top_json[0]["orgId"]
cninfo_600081_org_id = cninfo_600081_top_json[0]["orgId"]
cninfo_688603_org_id = cninfo_688603_top_json[0]["orgId"]
cninfo_600519_org_id = cninfo_600519_top_json[0]["orgId"]
cninfo_600497_org_id = cninfo_600497_top_json[0]["orgId"]
cninfo_000539_org_id = cninfo_000539_top_json[0]["orgId"]

cninfo_org_ids = pd.Series(
    {
        "603888 新华网": cninfo_603888_org_id,
        "600081 东风科技": cninfo_600081_org_id,
        "688603 天承科技": cninfo_688603_org_id,
        "600519 贵州茅台": cninfo_600519_org_id,
        "600497 驰宏锌锗": cninfo_600497_org_id,
        "000539 粤电力 A": cninfo_000539_org_id,
    },
    name="orgId",
)
cninfo_org_ids

603888 新华网       9900024951
600081 东风科技     gssh0600081
688603 天承科技      9900048047
600519 贵州茅台     gssh0600519
600497 驰宏锌锗     gssh0600497
000539 粤电力 A    gssz0000539
Name: orgId, dtype: str

## 普通分红、送股、转增

对 `10 派 1`、`10 送 1` 这类事件，先检索 `权益分派实施公告`。返回记录通常已经包含公告日期、公告编号和公告文件地址；具体的现金、送股、转增比例以及除权除息日写在公告正文里。

In [11]:
cninfo_603888_dividend_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"603888,{cninfo_603888_org_id}",
        "tabName": "fulltext",
        "searchkey": "权益分派实施公告",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_603888_dividend_resp.raise_for_status()
cninfo_603888_dividend_payload = cninfo_603888_dividend_resp.json()
cninfo_603888_dividend_payload["totalAnnouncement"]

16

In [12]:
cninfo_603888_dividend_records = (
    cninfo_603888_dividend_payload.get("announcements") or []
)
cninfo_603888_dividend = pd.DataFrame(
    cninfo_603888_dividend_records,
    columns=[
        "secCode",
        "secName",
        "announcementTitle",
        "announcementTime",
        "announcementId",
        "adjunctUrl",
        "adjunctType",
    ],
)
cninfo_603888_dividend["公告日期"] = pd.to_datetime(
    cninfo_603888_dividend["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_603888_dividend["标题"] = (
    cninfo_603888_dividend["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.replace("&nbsp;", " ", regex=False)
    .str.strip()
)
cninfo_603888_dividend["公告地址"] = (
    CNINFO_STATIC_BASE_URL
    + cninfo_603888_dividend["adjunctUrl"].fillna("")
)
cninfo_603888_dividend = cninfo_603888_dividend[
    ["secCode", "secName", "公告日期", "announcementId", "标题", "adjunctType", "公告地址"]
].rename(
    columns={"secCode": "代码", "secName": "简称", "announcementId": "公告编号", "adjunctType": "文件类型"}
)
cninfo_603888_dividend.head(20)

,代码,简称,公告日期,公告编号,标题,文件类型,公告地址
0,603888,新华网,2026-07-16,1225426539,新华网股份有限公司2025年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2026-07...
1,603888,新华网,2025-07-18,1224202430,新华网股份有限公司2024年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2025-07...
2,603888,新华网,2024-07-19,1220676697,新华网股份有限公司2023年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2024-07...
3,603888,新华网,2023-07-19,1217322314,新华网股份有限公司2022年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2023-07...
4,603888,新华网,2022-07-06,1213955304,新华网股份有限公司2021年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2022-07...
5,603888,新华网,2021-08-06,1210664454,新华网股份有限公司2020年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2021-08...
6,603888,新华网,2020-07-03,1207997083,2019年年度权益分派实施公告,TXT,https://static.cninfo.com.cn/finalpage/2020-07...
7,603888,新华网,2020-07-03,1207995497,2019年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2020-07...
8,603888,新华网,2019-07-23,1206468486,2018年年度权益分派实施公告,TXT,https://static.cninfo.com.cn/finalpage/2019-07...
9,603888,新华网,2019-07-23,1206468055,2018年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2019-07...


In [13]:
cninfo_603888_dividend_pdf_row = cninfo_603888_dividend.loc[
    cninfo_603888_dividend["公告地址"].str.contains(
        r"\.pdf$",
        case=False,
        na=False,
    )
].iloc[0]
cninfo_603888_dividend_pdf_url = cninfo_603888_dividend_pdf_row["公告地址"]
cninfo_603888_dividend_pdf_resp = client.get(
    cninfo_603888_dividend_pdf_url,
    headers=CNINFO_PDF_HEADERS,
)
cninfo_603888_dividend_pdf_resp.raise_for_status()
pd.Series(
    {
        "公告编号": cninfo_603888_dividend_pdf_row["公告编号"],
        "公告地址": cninfo_603888_dividend_pdf_url,
        "状态码": cninfo_603888_dividend_pdf_resp.status_code,
        "Content-Type": cninfo_603888_dividend_pdf_resp.headers.get("content-type"),
        "字节数": len(cninfo_603888_dividend_pdf_resp.content),
    }
)

公告编号                                                   1225426539
公告地址            https://static.cninfo.com.cn/finalpage/2026-07...
状态码                                                           200
Content-Type                                      application/pdf
字节数                                                        147977
dtype: object

## 配股

配股也通过公告检索获取。列表用于定位公告，PDF 正文里通常有每 10 股配售多少股、配股价格、股权登记日和配股除权日。下面使用 `600081` 的配股发行结果公告。

In [14]:
cninfo_600081_rights_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"600081,{cninfo_600081_org_id}",
        "tabName": "fulltext",
        "searchkey": "配股",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600081_rights_resp.raise_for_status()
cninfo_600081_rights_payload = cninfo_600081_rights_resp.json()
cninfo_600081_rights_records = (
    cninfo_600081_rights_payload.get("announcements") or []
)
cninfo_600081_rights = pd.DataFrame(
    cninfo_600081_rights_records,
    columns=[
        "secCode",
        "secName",
        "announcementTitle",
        "announcementTime",
        "announcementId",
        "adjunctUrl",
        "adjunctType",
    ],
)
cninfo_600081_rights["公告日期"] = pd.to_datetime(
    cninfo_600081_rights["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_600081_rights["标题"] = (
    cninfo_600081_rights["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.replace("&nbsp;", " ", regex=False)
    .str.strip()
)
cninfo_600081_rights["公告地址"] = (
    CNINFO_STATIC_BASE_URL
    + cninfo_600081_rights["adjunctUrl"].fillna("")
)
cninfo_600081_rights[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]].head(20)

,公告日期,announcementId,标题,adjunctType,公告地址
0,2025-04-29,1223361016,中信证券股份有限公司关于东风电子科技股份有限公司配股之持续督导保荐总结报告书,PDF,https://static.cninfo.com.cn/finalpage/2025-04...
1,2023-08-19,1217585363,东风电子科技股份有限公司配股股份变动及获配股票上市公告书,TXT,https://static.cninfo.com.cn/finalpage/2023-08...
2,2023-08-19,1217575254,中信证券股份有限公司关于东风电子科技股份有限公司配股之上市保荐书,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
3,2023-08-19,1217575253,东风电子科技股份有限公司配股股份变动及获配股票上市公告书,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
4,2023-08-10,1217501773,东风电子科技股份有限公司配股发行结果公告,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
5,2023-08-08,1217476955,东风电子科技股份有限公司配股提示性公告,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
6,2023-08-07,1217471700,东风电子科技股份有限公司配股提示性公告,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
7,2023-08-04,1217459026,东风电子科技股份有限公司配股提示性公告,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
8,2023-08-03,1217451704,东风电子科技股份有限公司配股提示性公告,PDF,https://static.cninfo.com.cn/finalpage/2023-08...
9,2023-08-02,1217442792,东风电子科技股份有限公司配股提示性公告,PDF,https://static.cninfo.com.cn/finalpage/2023-08...


In [15]:
cninfo_600081_rights_pdf_row = cninfo_600081_rights.loc[
    cninfo_600081_rights["标题"].str.contains(
        "配股发行结果公告",
        na=False,
    )
].loc[
    cninfo_600081_rights["公告地址"].str.contains(
        r"\.pdf$",
        case=False,
        na=False,
    )
].iloc[0]
cninfo_600081_rights_pdf_url = cninfo_600081_rights_pdf_row["公告地址"]
cninfo_600081_rights_pdf_resp = client.get(
    cninfo_600081_rights_pdf_url,
    headers=CNINFO_PDF_HEADERS,
)
cninfo_600081_rights_pdf_resp.raise_for_status()

cninfo_600081_rights_text = ""
if CNINFO_PDF_TEXT_AVAILABLE:
    with tempfile.TemporaryDirectory() as cninfo_600081_rights_tmp:
        cninfo_600081_rights_pdf_path = Path(cninfo_600081_rights_tmp) / "rights.pdf"
        cninfo_600081_rights_text_path = Path(cninfo_600081_rights_tmp) / "rights.txt"
        cninfo_600081_rights_pdf_path.write_bytes(cninfo_600081_rights_pdf_resp.content)
        subprocess.run(
            ["pdftotext", "-layout", str(cninfo_600081_rights_pdf_path), str(cninfo_600081_rights_text_path)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        cninfo_600081_rights_text = cninfo_600081_rights_text_path.read_text(encoding="utf-8", errors="replace")
cninfo_600081_rights_text[:2000] if cninfo_600081_rights_text else "当前环境未安装 pdftotext，已完成 PDF 下载"

'证券代码：600081         证券简称：东风科技              公告编号：2023-070\n\n\n                  东风电子科技股份有限公司\n                    配股发行结果公告\n           保荐人（主承销商）\n                   ：中信证券股份有限公司\n\n  本公司董事会及全体董事保证本公告内容不存在任何虚假记载、误导性陈述\n或者重大遗漏，并对其内容的真实性、准确性和完整性承担法律责任。\n\n\n\n  重要内容提示：\n\n  经中国证券监督管理委员会证监许可〔2023〕1370 号文同意注册，东风电\n子科技股份有限公司（以下简称“东风科技”、“公司”或“发行人”）向截至\n2023 年 8 月 1 日（T 日，股权登记日）上海证券交易所（以下简称“上交所”）\n收市后，在中国证券登记结算有限责任公司上海分公司登记在册的东风科技全体\n股东，按照每 10 股配售 3 股的比例向全体股东配售人民币普通股（A 股）。本次\n配股网上认购缴款工作已于 2023 年 8 月 8 日（T+5 日）结束。现将发行结果公\n告如下：\n\n\n一、认购情况\n\n  本次配股以股权登记日 2023 年 8 月 1 日（T 日）上交所收市后公司总股本\n447,276,315 股为基数，按每 10 股配售 3 股的比例向股权登记日全体股东配售，\n共计可配股票总数量为 134,182,894 股，均为无限售条件流通股。\n\n  东风科技本次配股采用网上定价发行方式，经上交所交易相关系统统计，并\n经中国证券登记结算有限责任公司上海分公司提供的网上认购数据验证，本次配\n股认购情况如下：\n\n 有效认购股数（股）          有效认购资金总额（元）          占可配售股份总数比例\n\n    131,067,214       1,256,934,582.26      97.68%\n\n\n二、发行结果\n                              1\n\x0c    根据本次配股发行公告，本次东风科技股东按照每股人民币 9.59 元的价格，\n以每 10 股配售 3 股的比例参与配售。本次东风科技配股共计可配售股份总数为\n13

In [16]:
cninfo_600081_rights_text_compact = re.sub(
    r"\s+",
    "",
    cninfo_600081_rights_text,
)
cninfo_600081_rights_ratio_match = re.search(
    r"每10股配售([\d.]+)股",
    cninfo_600081_rights_text_compact,
)
cninfo_600081_rights_price_match = re.search(
    r"每股人民币([\d.]+)元",
    cninfo_600081_rights_text_compact,
)
cninfo_600081_rights_record_date_match = re.search(
    r"截至(\d{4}年\d{1,2}月\d{1,2}日)（T日，股权登记日）",
    cninfo_600081_rights_text_compact,
)
cninfo_600081_rights_ex_date_match = re.search(
    r"(\d{4}年\d{1,2}月\d{1,2}日)[^。]{0,40}配股除权日",
    cninfo_600081_rights_text_compact,
)
cninfo_600081_rights_valid_number_match = re.search(
    r"有效认购数量为([\d,]+)股",
    cninfo_600081_rights_text_compact,
)
cninfo_600081_rights_summary = pd.Series(
    {
        "配股比例（每10股）": cninfo_600081_rights_ratio_match.group(1) if cninfo_600081_rights_ratio_match else pd.NA,
        "配股价格（元/股）": cninfo_600081_rights_price_match.group(1) if cninfo_600081_rights_price_match else pd.NA,
        "股权登记日": cninfo_600081_rights_record_date_match.group(1) if cninfo_600081_rights_record_date_match else pd.NA,
        "配股除权日": cninfo_600081_rights_ex_date_match.group(1) if cninfo_600081_rights_ex_date_match else pd.NA,
        "有效认购数量（股）": cninfo_600081_rights_valid_number_match.group(1) if cninfo_600081_rights_valid_number_match else pd.NA,
        "公告地址": cninfo_600081_rights_pdf_url,
    },
    name="600081 配股公告提取结果",
)
cninfo_600081_rights_summary

配股比例（每10股）                                                    3
配股价格（元/股）                                                  9.59
股权登记日                                                 2023年8月1日
配股除权日                                                2023年8月10日
有效认购数量（股）                                           131,067,214
公告地址          https://static.cninfo.com.cn/finalpage/2023-08...
Name: 600081 配股公告提取结果, dtype: str

## 增发和股本变动公告

巨潮没有和新浪 `vISSUE_AddStock` 完全对应的统一结构化页面；可以直接用公告全文检索。`增发` 适合定位历史增发类公告，`发行股份` 或 `股本变动` 往往更容易定位到发行结果和股本变化公告。

In [17]:
cninfo_000539_add_stock_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"000539,{cninfo_000539_org_id}",
        "tabName": "fulltext",
        "searchkey": "增发",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_000539_add_stock_resp.raise_for_status()
cninfo_000539_add_stock_payload = cninfo_000539_add_stock_resp.json()
cninfo_000539_add_stock_records = cninfo_000539_add_stock_payload.get("announcements") or []
cninfo_000539_add_stock = pd.DataFrame(cninfo_000539_add_stock_records)
cninfo_000539_add_stock["公告日期"] = pd.to_datetime(
    cninfo_000539_add_stock["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_000539_add_stock["标题"] = (
    cninfo_000539_add_stock["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
)
cninfo_000539_add_stock["公告地址"] = CNINFO_STATIC_BASE_URL + cninfo_000539_add_stock["adjunctUrl"].fillna("")
cninfo_000539_add_stock[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]].head(20)

,公告日期,announcementId,标题,adjunctType,公告地址
0,2003-05-15 05:07:11,10704093,粤电力Ａ2001年Ａ股增发的第二次回访报告（海通证券）,PDF,https://static.cninfo.com.cn/finalpage/2003-05...


In [18]:
cninfo_600497_share_change_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"600497,{cninfo_600497_org_id}",
        "tabName": "fulltext",
        "searchkey": "股本变动",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600497_share_change_resp.raise_for_status()
cninfo_600497_share_change_payload = cninfo_600497_share_change_resp.json()
cninfo_600497_share_change_records = cninfo_600497_share_change_payload.get("announcements") or []
cninfo_600497_share_change = pd.DataFrame(cninfo_600497_share_change_records)
cninfo_600497_share_change["公告日期"] = pd.to_datetime(
    cninfo_600497_share_change["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_600497_share_change["标题"] = (
    cninfo_600497_share_change["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
)
cninfo_600497_share_change["公告地址"] = CNINFO_STATIC_BASE_URL + cninfo_600497_share_change["adjunctUrl"].fillna("")
cninfo_600497_share_change[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]].head(20)

,公告日期,announcementId,标题,adjunctType,公告地址
0,2017-12-02 00:00:00,1204185804,关于2016年度非公开发行股票发行结果暨股本变动的公告,PDF,https://static.cninfo.com.cn/finalpage/2017-12...
1,2016-05-07 00:00:00,1202303036,关于发行股份及支付现金购买资产并募集配套资金暨关联交易之发行结果暨股本变动的补充更正公告,PDF,https://static.cninfo.com.cn/finalpage/2016-05...
2,2016-05-06 00:00:00,1202299517,发行股份及支付现金购买资产并募集配套资金暨关联交易之发行结果暨股本变动公告,PDF,https://static.cninfo.com.cn/finalpage/2016-05...
3,2016-04-02 00:00:00,1202133751,发行股份购买资产发行结果暨股本变动公告,PDF,https://static.cninfo.com.cn/finalpage/2016-04...
4,2009-12-14 06:35:00,57394386,2009年度配股股份上市及股本变动公告,PDF,https://static.cninfo.com.cn/finalpage/2009-12...
5,2009-12-14 05:45:00,57394356,2009年度配股股份上市及股本变动公告,NaN,https://static.cninfo.com.cn/finalpage/2009-12...


In [19]:
cninfo_600497_share_change_pdf_row = cninfo_600497_share_change.loc[
    cninfo_600497_share_change["标题"].str.contains(
        "2016年度非公开发行股票发行结果暨股本变动的公告",
        na=False,
    )
].loc[
    cninfo_600497_share_change["公告地址"].str.contains(
        r"\.pdf$",
        case=False,
        na=False,
    )
].iloc[0]
cninfo_600497_share_change_pdf_url = cninfo_600497_share_change_pdf_row["公告地址"]
cninfo_600497_share_change_pdf_resp = client.get(
    cninfo_600497_share_change_pdf_url,
    headers=CNINFO_PDF_HEADERS,
)
cninfo_600497_share_change_pdf_resp.raise_for_status()
pd.Series(
    {
        "公告编号": cninfo_600497_share_change_pdf_row["announcementId"],
        "公告地址": cninfo_600497_share_change_pdf_url,
        "状态码": cninfo_600497_share_change_pdf_resp.status_code,
        "字节数": len(cninfo_600497_share_change_pdf_resp.content),
    }
)

公告编号                                           1204185804
公告地址    https://static.cninfo.com.cn/finalpage/2017-12...
状态码                                                   200
字节数                                                442604
dtype: object

## 差异化分红

下面从 `688603` 的 2024 年年度权益分派实施公告中读取差异化分红需要的数字。当前环境用系统 `pdftotext` 将 PDF 转成文本；如果换到没有这个命令的环境，前面的 PDF 下载仍然可以执行，只是文本提取单元会显示提示。

In [20]:
cninfo_688603_dividend_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"688603,{cninfo_688603_org_id}",
        "tabName": "fulltext",
        "searchkey": "权益分派实施公告",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_688603_dividend_resp.raise_for_status()
cninfo_688603_dividend_payload = cninfo_688603_dividend_resp.json()
cninfo_688603_dividend_records = cninfo_688603_dividend_payload.get("announcements") or []
cninfo_688603_dividend = pd.DataFrame(cninfo_688603_dividend_records)
cninfo_688603_dividend["公告日期"] = pd.to_datetime(
    cninfo_688603_dividend["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_688603_dividend["标题"] = (
    cninfo_688603_dividend["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
)
cninfo_688603_dividend["公告地址"] = CNINFO_STATIC_BASE_URL + cninfo_688603_dividend["adjunctUrl"].fillna("")
cninfo_688603_dividend[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]]

,公告日期,announcementId,标题,adjunctType,公告地址
0,2026-06-23,1225381292,2025年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2026-06...
1,2025-06-13,1223860268,2024年年度权益分派实施结果暨股份上市公告,PDF,https://static.cninfo.com.cn/finalpage/2025-06...
2,2025-06-07,1223801462,2024年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2025-06...
3,2025-01-17,1222353973,关于2024年前三季度权益分派实施后调整回购股份价格上限的公告,PDF,https://static.cninfo.com.cn/finalpage/2025-01...
4,2025-01-06,1222238925,2024年前三季度权益分派实施结果暨股份上市公告,PDF,https://static.cninfo.com.cn/finalpage/2025-01...
5,2024-12-27,1222154646,广东天承科技股份有限公司2024年前三季度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2024-12...
6,2024-06-22,1220428345,关于2023年年度权益分派实施后调整回购股份价格上限的公告,PDF,https://static.cninfo.com.cn/finalpage/2024-06...
7,2024-06-15,1220359259,2023年年度权益分派实施公告,PDF,https://static.cninfo.com.cn/finalpage/2024-06...


In [21]:
cninfo_688603_differential_pdf_row = cninfo_688603_dividend.loc[
    cninfo_688603_dividend["标题"].str.contains(
        "2024年年度权益分派实施公告",
        na=False,
    )
].loc[
    cninfo_688603_dividend["公告地址"].str.contains(
        r"\.pdf$",
        case=False,
        na=False,
    )
].iloc[0]
cninfo_688603_differential_pdf_url = cninfo_688603_differential_pdf_row["公告地址"]
cninfo_688603_differential_pdf_resp = client.get(
    cninfo_688603_differential_pdf_url,
    headers=CNINFO_PDF_HEADERS,
)
cninfo_688603_differential_pdf_resp.raise_for_status()

cninfo_688603_differential_text = ""
if CNINFO_PDF_TEXT_AVAILABLE:
    with tempfile.TemporaryDirectory() as cninfo_688603_differential_tmp:
        cninfo_688603_differential_pdf_path = Path(cninfo_688603_differential_tmp) / "differential.pdf"
        cninfo_688603_differential_text_path = Path(cninfo_688603_differential_tmp) / "differential.txt"
        cninfo_688603_differential_pdf_path.write_bytes(cninfo_688603_differential_pdf_resp.content)
        subprocess.run(
            ["pdftotext", "-layout", str(cninfo_688603_differential_pdf_path), str(cninfo_688603_differential_text_path)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        cninfo_688603_differential_text = cninfo_688603_differential_text_path.read_text(encoding="utf-8", errors="replace")
cninfo_688603_differential_text[:2200] if cninfo_688603_differential_text else "当前环境未安装 pdftotext，已完成 PDF 下载"

'证券代码：688603           证券简称：天承科技             公告编号：2025-028\n\n  广东天承科技股份有限公司2024年年度权益分派实施公告\n\n  本公司董事会及全体董事保证公告内容不存在任何虚假记载、误导性陈述或\n者重大遗漏，并对其内容的真实性、准确性和完整性依法承担法律责任。\n\n重要内容提示：\n\n  \uf06c 公司存在首发战略配售股份，首发战略配售股份未全部上市流通\n  \uf06c 是否涉及差异化分红送转：是\n  \uf06c 每股分配比例，每股转增比例\n      每股现金红利0.3元（含税）\n      每股转增0.49股\n  \uf06c 相关日期\n                              新增无限售条件流通\n  股权登记日           除权（息）日                      现金红利发放日\n                                股份上市日\n   2025/6/12      2025/6/13     2025/6/13       2025/6/13\n\n\n一、通过分配、转增股本方案的股东会届次和日期\n\n  本次利润分配及转增股本方案经公司2025 年 4 月 30 日的2024年年度股东会审\n议通过。\n\n二、分配、转增股本方案\n\n  1、发放年度：2024年年度\n  2、分派对象：\n  截至股权登记日下午上海证券交易所收市后，在中国证券登记结算有限责任\n公司上海分公司（以下简称“中国结算上海分公司”）登记在册的本公司全体股东。\n（公司回购专用证券账户除外）。\n  根据《中华人民共和国公司法》《中华人民共和国证券法》《上海证券交易所\n上市公司自律监管指引第 7 号——回购股份》等相关法律、行政法规、部门规章\n及其他规范性文件以及《公司章程》的有关规定，公司回购专用证券账户中的股\n份不享有股东会表决权、利润分配、公积金转增股本、认购新股等权利。\n\x0c   3、差异化分红送转方案：\n   （1）本次差异化分红、转增方案\n   根据公司 2024 年年度股东会审议通过的《关于 2024 年度利润分配预案的议\n案》，公司 2024 年年度利润分配方案如下：\n   1）

In [22]:
cninfo_688603_differential_text_compact = re.sub(
    r"\s+",
    "",
    cninfo_688603_differential_text,
)
cninfo_688603_total_shares_match = re.search(
    r"公司总股本为([\d,]+)股",
    cninfo_688603_differential_text_compact,
)
cninfo_688603_participating_shares_match = re.search(
    r"扣除公司回购专用证券账户中股份数[\d,]+股后的股本数为([\d,]+)股",
    cninfo_688603_differential_text_compact,
)
cninfo_688603_actual_cash_match = re.search(
    r"每股现金红利([\d.]+)元",
    cninfo_688603_differential_text_compact,
)
cninfo_688603_actual_transfer_match = re.search(
    r"每股转增([\d.]+)股",
    cninfo_688603_differential_text_compact,
)
cninfo_688603_virtual_cash_match = re.search(
    r"虚拟分派的每股现金红利=.*?≈([\d.]+)元/股",
    cninfo_688603_differential_text_compact,
)
cninfo_688603_virtual_transfer_match = re.search(
    r"流通股份变动比例=.*?≈([\d.]+)",
    cninfo_688603_differential_text_compact,
)
cninfo_688603_total_shares = (
    int(cninfo_688603_total_shares_match.group(1).replace(",", ""))
    if cninfo_688603_total_shares_match
    else pd.NA
)
cninfo_688603_participating_shares = (
    int(cninfo_688603_participating_shares_match.group(1).replace(",", ""))
    if cninfo_688603_participating_shares_match
    else pd.NA
)
cninfo_688603_actual_cash_per_share = (
    float(cninfo_688603_actual_cash_match.group(1))
    if cninfo_688603_actual_cash_match
    else pd.NA
)
cninfo_688603_actual_transfer_per_share = (
    float(cninfo_688603_actual_transfer_match.group(1))
    if cninfo_688603_actual_transfer_match
    else pd.NA
)
cninfo_688603_computed_virtual_cash = (
    cninfo_688603_participating_shares
    * cninfo_688603_actual_cash_per_share
    / cninfo_688603_total_shares
    if pd.notna(cninfo_688603_total_shares) and pd.notna(cninfo_688603_participating_shares) and pd.notna(cninfo_688603_actual_cash_per_share)
    else pd.NA
)
cninfo_688603_computed_virtual_transfer = (
    cninfo_688603_participating_shares
    * cninfo_688603_actual_transfer_per_share
    / cninfo_688603_total_shares
    if pd.notna(cninfo_688603_total_shares) and pd.notna(cninfo_688603_participating_shares) and pd.notna(cninfo_688603_actual_transfer_per_share)
    else pd.NA
)
cninfo_688603_differential_summary = pd.Series(
    {
        "总股本（股）": cninfo_688603_total_shares,
        "参与分配股本（股）": cninfo_688603_participating_shares,
        "实际现金红利（元/股）": cninfo_688603_actual_cash_per_share,
        "实际转增比例（股/股）": cninfo_688603_actual_transfer_per_share,
        "公告虚拟现金红利（元/股）": cninfo_688603_virtual_cash_match.group(1) if cninfo_688603_virtual_cash_match else pd.NA,
        "公告流通股份变动比例": cninfo_688603_virtual_transfer_match.group(1) if cninfo_688603_virtual_transfer_match else pd.NA,
        "按股本计算虚拟现金红利": cninfo_688603_computed_virtual_cash,
        "按股本计算虚拟流通股份变动比例": cninfo_688603_computed_virtual_transfer,
        "复权输入公式": "（前收盘价-虚拟现金红利）÷（1+虚拟流通股份变动比例）",
        "公告地址": cninfo_688603_differential_pdf_url,
    },
    name="688603 差异化分红提取结果",
)
cninfo_688603_differential_summary

总股本（股）                                                      83957192
参与分配股本（股）                                                   83198636
实际现金红利（元/股）                                                      0.3
实际转增比例（股/股）                                                     0.49
公告虚拟现金红利（元/股）                                                 0.2973
公告流通股份变动比例                                                    0.4856
按股本计算虚拟现金红利                                                 0.297289
按股本计算虚拟流通股份变动比例                                             0.485573
复权输入公式                                  （前收盘价-虚拟现金红利）÷（1+虚拟流通股份变动比例）
公告地址               https://static.cninfo.com.cn/finalpage/2025-06...
Name: 688603 差异化分红提取结果, dtype: object

## 股权分置改革、现金/股票对价和权证

股权分置不是普通的 `10 派 1` 或 `10 送 1`。巨潮公告原文可以给出转增、流通股对价股份、现金对价、权证数量、实施日期，以及哪些对价不计入除权参考价。下面使用 `600519` 的股权分置改革方案实施公告。

In [23]:
cninfo_600519_reform_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"600519,{cninfo_600519_org_id}",
        "tabName": "fulltext",
        "searchkey": "股权分置",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600519_reform_resp.raise_for_status()
cninfo_600519_reform_payload = cninfo_600519_reform_resp.json()
cninfo_600519_reform_records = cninfo_600519_reform_payload.get("announcements") or []
cninfo_600519_reform = pd.DataFrame(cninfo_600519_reform_records)
cninfo_600519_reform["公告日期"] = pd.to_datetime(
    cninfo_600519_reform["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_600519_reform["标题"] = (
    cninfo_600519_reform["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
)
cninfo_600519_reform["公告地址"] = CNINFO_STATIC_BASE_URL + cninfo_600519_reform["adjunctUrl"].fillna("")
cninfo_600519_reform[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]].head(30)

,公告日期,announcementId,标题,adjunctType,公告地址
0,2006-05-22 07:05:00,17136922,股权分置改革方案实施公告,PDF,https://static.cninfo.com.cn/finalpage/2006-05...
1,2006-05-22 05:35:00,17132664,股权分置改革方案实施公告,NaN,https://static.cninfo.com.cn/finalpage/2006-05...
2,2006-05-09 07:02:00,17034041,关于股权分置改革方案获贵州省国有资产监督管理委员会批准的公告,PDF,https://static.cninfo.com.cn/finalpage/2006-05...
3,2006-05-09 07:02:00,17033842,关于股权分置改革方案获贵州省国资委批准的公告,NaN,https://static.cninfo.com.cn/finalpage/2006-05...
4,2006-04-15 05:50:00,16912804,股权分置改革方案股东沟通协商结果暨调整股权分置改革方案勘误公告,NaN,https://static.cninfo.com.cn/finalpage/2006-04...
5,2006-04-15 05:35:00,16912442,关于股权分置改革方案股东沟通协商结果暨调整股权分置改革方案勘误公告,PDF,https://static.cninfo.com.cn/finalpage/2006-04...
6,2006-04-14 05:50:00,16901971,股权分置改革事项的补充法律意见书,PDF,https://static.cninfo.com.cn/finalpage/2006-04...
7,2006-04-14 05:50:00,16901970,股权分置改革说明书（修订稿）,PDF,https://static.cninfo.com.cn/finalpage/2006-04...
8,2006-04-14 05:45:00,16901974,股权分置改革说明书摘要（修订稿）,PDF,https://static.cninfo.com.cn/finalpage/2006-04...
9,2006-04-14 05:40:00,16901973,股权分置改革之补充保荐意见,PDF,https://static.cninfo.com.cn/finalpage/2006-04...


In [24]:
cninfo_600519_reform_pdf_row = cninfo_600519_reform.loc[
    cninfo_600519_reform["标题"].str.contains(
        "股权分置改革方案实施公告",
        na=False,
    )
].loc[
    cninfo_600519_reform["公告地址"].str.contains(
        r"\.pdf$",
        case=False,
        na=False,
    )
].iloc[0]
cninfo_600519_reform_pdf_url = cninfo_600519_reform_pdf_row["公告地址"]
cninfo_600519_reform_pdf_resp = client.get(
    cninfo_600519_reform_pdf_url,
    headers=CNINFO_PDF_HEADERS,
)
cninfo_600519_reform_pdf_resp.raise_for_status()

cninfo_600519_reform_text = ""
if CNINFO_PDF_TEXT_AVAILABLE:
    with tempfile.TemporaryDirectory() as cninfo_600519_reform_tmp:
        cninfo_600519_reform_pdf_path = Path(cninfo_600519_reform_tmp) / "reform.pdf"
        cninfo_600519_reform_text_path = Path(cninfo_600519_reform_tmp) / "reform.txt"
        cninfo_600519_reform_pdf_path.write_bytes(cninfo_600519_reform_pdf_resp.content)
        subprocess.run(
            ["pdftotext", "-layout", str(cninfo_600519_reform_pdf_path), str(cninfo_600519_reform_text_path)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        cninfo_600519_reform_text = cninfo_600519_reform_text_path.read_text(encoding="utf-8", errors="replace")
cninfo_600519_reform_text[:2400] if cninfo_600519_reform_text else "当前环境未安装 pdftotext，已完成 PDF 下载"

'证券简称：贵州茅台                 证券代码：600519      编号：临 2006-029\n\n\n\n           贵州茅台酒股份有限公司\n           股权分置改革方案实施公告\n\n                  特别提示\n  公司及董事会全体成员保证本公告内容的真实、准确和完整，对本公告的虚假记载、\n误导性陈述或者重大遗漏负连带责任。\n\n\n\n                          重要内容提示：\n\n● 作为公司股权分置改革方案的一部分，公司以 2006 年 5 月 18 日为资本公积金转增股本股\n\n权登记日，已向该日登记在册的公司全体股东实施了资本公积金每 10 股转增 10 股。\n\n● 公司非流通股股东向本次方案实施股权登记日在登记公司登记在册的公司全体流通股股东\n\n对价安排由以下三部分构成：\n\n   1、公司以转增后的总股本 943,800,000 股为基数，向全体股东每 10 股分派现金 5.91 元（含\n\n税），非流通股股东将其应得的现金股利全部执行给流通股股东，则每 10 股流通股股份实际\n\n得到 20.66 元现金（含税），其中 5.91 元为流通股股东应得的现金股利，14.75 元为非流通股\n\n股东的对价安排。\n\n   2、公司全体非流通股股东以转增后流通股股本 269,926,800 股为基数，向全体流通股股\n\n东每 10 股支付 1.2 股股份。\n\n   3、中国贵州茅台酒厂有限责任公司以转增后流通股股本 269,926,800 股为基数，向全体\n\n流通股股东按转增后股本每 10 股无偿派发 16 份存续期限为 12 个月，行权比例为 4:1 的欧式\n\n认沽权证。权证的初始行权价为 30.30 元，行权期间为权证存续期的最后 1 个交易日。\n\n● 方案实施股权登记日：2006 年 5 月 23 日\n\n● 除权除息日：2006 年 5 月 24 日\n\n● 股票复牌及新增可流通股上市日：2006 年 5 月 25 日\n\n 本日股价不设涨跌幅限制，不纳入指数计算\n\n● 自 2006 年 5 月 25 日起，公司股票简称由“贵州茅台”变更为“G 茅台”，股票代码“600519”\n\n保持不变\n\x0c 

In [25]:
cninfo_600519_reform_text_compact = re.sub(
    r"\s+",
    "",
    cninfo_600519_reform_text,
)
cninfo_600519_transfer_match = re.search(
    r"每10股转增([\d.]+)股",
    cninfo_600519_reform_text_compact,
)
cninfo_600519_cash_consideration_match = re.search(
    r"每10股流通股(?:股份)?实际得到([\d.]+)元现金",
    cninfo_600519_reform_text_compact,
)
cninfo_600519_stock_consideration_match = re.search(
    r"每10股支付([\d.]+)股股份",
    cninfo_600519_reform_text_compact,
)
cninfo_600519_warrant_match = re.search(
    r"每10股无偿派发([\d,]+)份",
    cninfo_600519_reform_text_compact,
)
cninfo_600519_record_date_match = re.search(
    r"方案实施股权登记日[：:](\d{4}年\d{1,2}月\d{1,2}日)",
    cninfo_600519_reform_text_compact,
)
cninfo_600519_ex_date_match = re.search(
    r"除权除息日[：:](\d{4}年\d{1,2}月\d{1,2}日)",
    cninfo_600519_reform_text_compact,
)
cninfo_600519_reform_summary = pd.Series(
    {
        "资本公积转增（每10股）": cninfo_600519_transfer_match.group(1) if cninfo_600519_transfer_match else pd.NA,
        "流通股实际现金对价（元/每10股）": cninfo_600519_cash_consideration_match.group(1) if cninfo_600519_cash_consideration_match else pd.NA,
        "流通股股票对价（股/每10股）": cninfo_600519_stock_consideration_match.group(1) if cninfo_600519_stock_consideration_match else pd.NA,
        "认沽权证（份/每10股）": cninfo_600519_warrant_match.group(1) if cninfo_600519_warrant_match else pd.NA,
        "方案实施股权登记日": cninfo_600519_record_date_match.group(1) if cninfo_600519_record_date_match else pd.NA,
        "除权除息日": cninfo_600519_ex_date_match.group(1) if cninfo_600519_ex_date_match else pd.NA,
        "公告是否说明对价不计入除权参考价": "不计入公司股票的除权参考价" in cninfo_600519_reform_text_compact,
        "公告地址": cninfo_600519_reform_pdf_url,
    },
    name="600519 股权分置公告提取结果",
)
cninfo_600519_reform_summary

资本公积转增（每10股）                                                        10
流通股实际现金对价（元/每10股）                                                20.66
流通股股票对价（股/每10股）                                                    1.2
认沽权证（份/每10股）                                                        16
方案实施股权登记日                                                   2006年5月23日
除权除息日                                                       2006年5月24日
公告是否说明对价不计入除权参考价                                                  True
公告地址                 https://static.cninfo.com.cn/finalpage/2006-05...
Name: 600519 股权分置公告提取结果, dtype: object

## 回购注销、股本变动和宽关键词检索

缩股、合股、并股等特殊变化未必使用固定标题，也未必真的出现“缩股”这两个字。巨潮的做法是先用 `股本变动`、`减资`、`回购注销`、`股权分置`、`发行股份` 等宽关键词找到公告，再从公告正文读取股本前后数量和生效日期。下面先展示 `600081` 的回购注销和股本变动公告。

In [26]:
cninfo_600081_repurchase_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"600081,{cninfo_600081_org_id}",
        "tabName": "fulltext",
        "searchkey": "回购注销",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600081_repurchase_resp.raise_for_status()
cninfo_600081_repurchase_payload = cninfo_600081_repurchase_resp.json()
cninfo_600081_repurchase_records = cninfo_600081_repurchase_payload.get("announcements") or []
cninfo_600081_repurchase = pd.DataFrame(cninfo_600081_repurchase_records)
cninfo_600081_repurchase["公告日期"] = pd.to_datetime(
    cninfo_600081_repurchase["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_600081_repurchase["标题"] = (
    cninfo_600081_repurchase["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
)
cninfo_600081_repurchase["公告地址"] = CNINFO_STATIC_BASE_URL + cninfo_600081_repurchase["adjunctUrl"].fillna("")
cninfo_600081_repurchase[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]]

,公告日期,announcementId,标题,adjunctType,公告地址
0,2024-07-11,1220602950,东风电子科技股份有限公司关于回购并注销业绩补偿股份实施结果暨股份变动的公告,PDF,https://static.cninfo.com.cn/finalpage/2024-07...
1,2024-04-16,1219619325,东风电子科技股份有限公司关于回购注销部分股票减少注册资本暨通知债权人的公告,PDF,https://static.cninfo.com.cn/finalpage/2024-04...
2,2024-03-30,1219466771,东风电子科技股份有限公司关于重大资产重组业绩承诺补偿方案暨回购注销对应补偿股份的公告,PDF,https://static.cninfo.com.cn/finalpage/2024-03...
3,2023-07-14,1217292986,东风电子科技股份有限公司关于回购并注销业绩补偿股份实施结果暨股份变动的公告,PDF,https://static.cninfo.com.cn/finalpage/2023-07...
4,2023-04-27,1216614883,东风电子科技股份有限公司关于回购注销部分股票减少注册资本暨通知债权人的公告,PDF,https://static.cninfo.com.cn/finalpage/2023-04...
5,2023-04-11,1216370243,东风电子科技股份有限公司关于重大资产重组业绩承诺补偿方案暨回购注销对应补偿股份的公告,PDF,https://static.cninfo.com.cn/finalpage/2023-04...


In [ ]:
cninfo_600081_capital_change_resp = client.post(
    CNINFO_ANNOUNCEMENT_ENDPOINT,
    data={
        "stock": f"600081,{cninfo_600081_org_id}",
        "tabName": "fulltext",
        "searchkey": "股本变动",
        "seDate": CNINFO_DATE_RANGE,
        "pageNum": 1,
        "pageSize": 50,
        "isHLtitle": "true",
    },
    headers=CNINFO_REQUEST_HEADERS,
)
cninfo_600081_capital_change_resp.raise_for_status()
cninfo_600081_capital_change_payload = cninfo_600081_capital_change_resp.json()
cninfo_600081_capital_change_records = cninfo_600081_capital_change_payload.get("announcements") or []
cninfo_600081_capital_change = pd.DataFrame(cninfo_600081_capital_change_records)
cninfo_600081_capital_change["公告日期"] = pd.to_datetime(
    cninfo_600081_capital_change["announcementTime"],
    unit="ms",
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
cninfo_600081_capital_change["标题"] = (
    cninfo_600081_capital_change["announcementTitle"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
)
cninfo_600081_capital_change["公告地址"] = CNINFO_STATIC_BASE_URL + cninfo_600081_capital_change["adjunctUrl"].fillna("")
cninfo_600081_capital_change[["公告日期", "announcementId", "标题", "adjunctType", "公告地址"]]

## 结论

- 巨潮可以获取普通分红、送股、转增、配股、增发/发行股份、回购注销和股权分置等事件的公告记录与原始文件。
- 普通 `10 派 1`、`10 送 1` 只要拿到实施公告、方案数字和日期，就足够作为自算复权价的输入。
- 差异化分红可以从实施公告中拿到总股本、参与分配股本、实际每股分配和虚拟摊薄参数。
- 缩股、合股、并股等特殊变化，巨潮通常不是统一结构化字段；可以获取公告原文，但需要扩大关键词并从正文读取具体比例、股本前后数量和生效日期。
- 所以巨潮适合作为新浪之外的原始公告备源，但不是“无需解析、直接返回完整复权因子”的接口。